# Lecture 1: XAI Foundations & Interpretable Models
**Dataset:** scikit-learn breast cancer (binary: malignant vs benign).
**Goal:** read what interpretable models tell us, and see the accuracy–interpretability tradeoff.
Cells marked `TODO` are your tasks.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target   # note: target 1 = benign, 0 = malignant
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(X.shape); X.head()

## Part 1: Logistic regression and odds ratios
Standardize features so coefficients are comparable. Then `exp(coef)` is the odds ratio per one standard deviation.

In [ ]:
scaler = StandardScaler().fit(X_train)
Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)
logreg = LogisticRegression(max_iter=1000).fit(Xtr, y_train)
print("Test accuracy:", accuracy_score(y_test, logreg.predict(Xte)))

coefs = pd.Series(logreg.coef_[0], index=X.columns).sort_values()
pd.concat([coefs.head(5), coefs.tail(5)]).plot.barh(figsize=(7,4), title="Largest coefficients")
plt.show()

**TODO 1.** Compute the odds ratio for the 5 features with the largest absolute coefficient and interpret one of them in a sentence. Remember that target 1 means *benign*.

In [ ]:
# TODO 1: odds ratios
odds_ratios = ...  # hint: np.exp(...)
# odds_ratios

**TODO 2 (pitfall).** Many features here are highly correlated (e.g. radius, perimeter, area). Check the correlation of `mean radius` with `mean perimeter` and `mean area`, then refit with only one of the three. Do the coefficients change? What does this mean for interpretation?

In [ ]:
# TODO 2
# X[['mean radius','mean perimeter','mean area']].corr()

## Part 2: Decision trees and the depth tradeoff

In [ ]:
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train)); test_acc.append(t.score(X_test, y_test))
plt.plot(depths, train_acc, label="train"); plt.plot(depths, test_acc, label="test")
plt.xlabel("max_depth"); plt.ylabel("accuracy"); plt.legend(); plt.show()

**TODO 3.** Pick the *smallest* depth whose test accuracy is within 1 percentage point of the best. Justify the choice in terms of interpretability.
Then compare against a random forest baseline.

In [ ]:
# TODO 3
best_depth = ...
tree = DecisionTreeClassifier(max_depth=best_depth, random_state=0).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
# print accuracies of tree, logreg and rf

## Part 3: Reading the rules of a tree

In [ ]:
small = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(small, feature_names=list(X.columns)))
plt.figure(figsize=(14,6)); plot_tree(small, feature_names=X.columns, class_names=["malignant","benign"], filled=True); plt.show()

**TODO 4.** Pick one test sample and write down the path it follows through `small`, as plain-language rules. Use `small.decision_path(...)`, or trace it by hand from the printout.

In [ ]:
# TODO 4
sample = X_test.iloc[[0]]
# small.predict(sample), small.decision_path(sample)

## Part 4: Instability of trees
Train the same depth-3 tree on 5 bootstrap samples and look at the root split.

In [ ]:
rng = np.random.RandomState(0)
for i in range(5):
    idx = rng.choice(len(X_train), len(X_train), replace=True)
    t = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train.iloc[idx], y_train.iloc[idx])
    root = t.tree_.feature[0]
    print(f"Bootstrap {i}: root split on '{X.columns[root]}' at {t.tree_.threshold[0]:.2f}, test acc {t.score(X_test, y_test):.3f}")

**TODO 5.** Compare the coefficient stability of logistic regression with the tree on the same bootstrap samples. Which model is more stable, and why?

## Discussion
1. When would you accept a less accurate but interpretable model?
2. Which of the models here is *intrinsically* interpretable, and what does it actually tell you (global or local)?
3. Which GDPR / EU AI Act requirements would apply if this model supported a clinical decision?